# Anomaly Detection from Scratch: Isolation Forest, One-Class SVM & Autoencoders

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/supervised/anomaly_detection_from_scratch.ipynb)

Companion notebook to [Anomaly Detection from Scratch](https://sesen.ai/blog/anomaly-detection-from-scratch).

Three unsupervised detectors on real credit-card fraud (0.17% positives):

1. **Isolation Forest** built from scratch in NumPy (random-split trees + path-length scoring).
2. **One-Class SVM** boundary (scikit-learn), trained on legitimate transactions only.
3. **Autoencoder** (PyTorch) scored by reconstruction error.

We compare them with one honest ROC / precision-recall panel and expose the metric trap that makes every detector look near-perfect under extreme imbalance.

## Setup

In [ ]:
# On Colab, torch and scikit-learn are preinstalled. Otherwise:
# !pip install scikit-learn matplotlib numpy torch scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)

TEAL, AMBER, VIOLET = "#128173", "#C0872B", "#7E5BB5"
NORMAL, ANOM = "#9AA3AB", "#C0392B"
rng = np.random.default_rng(0)

## 1. The data: credit-card fraud

The ULB credit-card dataset: 284,807 transactions, 492 fraud (0.172%). Features are
28 anonymised PCA components plus the amount. No labels are used for training, only for evaluation.

In [ ]:
df = fetch_openml(data_id=1597, as_frame=True).frame   # downloads ~67 MB once
y = df.pop("Class").astype(int).values
X = df.select_dtypes("number").values.astype(float)
print(f"X {X.shape}   fraud {y.sum()}   prevalence {y.mean():.4%}")

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, stratify=y, random_state=42)
scaler = StandardScaler().fit(Xtr)
Xtr, Xte = scaler.transform(Xtr), scaler.transform(Xte)
prevalence = yte.mean()
k = int(yte.sum())   # true anomalies in the test set

def prec_at_k(scores, ytrue, k):
    idx = np.argsort(scores)[::-1][:k]
    return ytrue[idx].sum() / k

def evaluate(name, scores):
    roc = roc_auc_score(yte, scores)
    pr  = average_precision_score(yte, scores)
    pk  = prec_at_k(scores, yte, k)
    print(f"{name:20s} ROC-AUC={roc:.3f}  PR-AUC={pr:.3f}  P@{k}={pk:.3f}  "
          f"(catches {int(pk*k)}/{k})")
    return roc, pr, pk

## 2. Isolation Forest from scratch

An isolation tree recursively cuts a random subsample on a random feature at a random
threshold. Anomalies sit apart, so a random cut fences them off in **fewer** splits.
The number of splits to isolate a point is the anomaly signal.

In [ ]:
class Node:
    __slots__ = ("feature", "thresh", "left", "right", "size", "depth", "leaf")

def build_tree(X, depth, height_limit, rng):
    node = Node(); n = len(X)
    if depth >= height_limit or n <= 1:
        node.leaf, node.size, node.depth = True, n, depth
        return node
    f = rng.integers(0, X.shape[1])
    lo, hi = X[:, f].min(), X[:, f].max()
    if lo == hi:
        node.leaf, node.size, node.depth = True, n, depth
        return node
    t = rng.uniform(lo, hi)
    mask = X[:, f] < t
    node.leaf, node.feature, node.thresh, node.depth = False, f, t, depth
    node.left  = build_tree(X[mask],  depth + 1, height_limit, rng)
    node.right = build_tree(X[~mask], depth + 1, height_limit, rng)
    return node

def fit_iforest(X, n_trees=100, psi=256, seed=0):
    rng = np.random.default_rng(seed)
    height_limit = int(np.ceil(np.log2(psi)))
    trees = [build_tree(X[rng.choice(len(X), min(psi, len(X)), replace=False)],
                        0, height_limit, rng) for _ in range(n_trees)]
    return trees, psi

EULER = 0.5772156649
def c_factor(n):
    return 2.0 * (np.log(n - 1) + EULER) - 2.0 * (n - 1) / n if n > 1 else 0.0

def _paths_one(node, X):
    out = np.empty(len(X)); stack = [(node, np.arange(len(X)))]
    while stack:
        nd, idx = stack.pop()
        if nd.leaf:
            out[idx] = nd.depth + c_factor(nd.size)     # add expected depth below a truncated leaf
        else:
            m = X[idx, nd.feature] < nd.thresh
            stack.append((nd.left, idx[m])); stack.append((nd.right, idx[~m]))
    return out

def iforest_paths(trees, X):
    S = np.zeros(len(X))
    for tr in trees:
        S += _paths_one(tr, X)
    return S / len(trees)

def anomaly_score(paths, psi):
    return 2.0 ** (-paths / c_factor(psi))

In [ ]:
trees, psi = fit_iforest(Xtr, n_trees=100, psi=256, seed=0)
if_paths  = iforest_paths(trees, Xte)
if_scores = anomaly_score(if_paths, psi)
if_roc, if_pr, if_pk = evaluate("IsolationForest", if_scores)

# sanity check against scikit-learn's implementation
skif = IsolationForest(n_estimators=100, max_samples=256, random_state=0).fit(Xtr)
sk_scores = -skif.score_samples(Xte)
from scipy.stats import spearmanr
print(f"scratch vs sklearn: ROC {roc_auc_score(yte, if_scores):.3f} vs "
      f"{roc_auc_score(yte, sk_scores):.3f}, score Spearman rho = "
      f"{spearmanr(if_scores, sk_scores).correlation:.3f}")

In [ ]:
# Path length -> score. Fraud isolates in fewer splits, so short paths become high scores.
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.hist(if_paths[yte==0], bins=60, color=NORMAL, alpha=.85, density=True, label="Legitimate")
a1.hist(if_paths[yte==1], bins=60, color=ANOM, alpha=.7, density=True, label="Fraud")
a1.set_xlabel("Average path length"); a1.set_title("Fraud isolates in fewer splits"); a1.legend()
a2.hist(if_scores[yte==0], bins=60, color=NORMAL, alpha=.85, density=True, label="Legitimate")
a2.hist(if_scores[yte==1], bins=60, color=ANOM, alpha=.7, density=True, label="Fraud")
a2.set_xlabel("Anomaly score"); a2.set_title("Short paths -> high scores"); a2.legend()
plt.tight_layout(); plt.show()

## 3. One-Class SVM: a boundary around the normal region

A One-Class SVM (Scholkopf 2001) / SVDD (Tax & Duin 2004) wraps the normal data in a
smooth kernel boundary and flags whatever falls outside. We train on legitimate transactions
only (novelty detection), on a subsample because the RBF kernel scales poorly.

In [ ]:
normals = Xtr[ytr == 0]
sub = rng.choice(len(normals), 6000, replace=False)
oc = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale").fit(normals[sub])
oc_scores = -oc.decision_function(Xte)          # outside the boundary = high
oc_roc, oc_pr, oc_pk = evaluate("OneClassSVM", oc_scores)

In [ ]:
# 2D illustration of the enclosing boundary
g = np.random.default_rng(1)
c1 = g.normal([-1.6,-1.4], .55, (120,2)); c2 = g.normal([1.7,1.5], .6, (120,2))
Xn2 = np.vstack([c1, c2]); an2 = g.uniform(-5, 5, (30,2))
an2 = an2[np.minimum(np.linalg.norm(an2-c1.mean(0),axis=1),
                     np.linalg.norm(an2-c2.mean(0),axis=1)) > 2.2]
oc2 = OneClassSVM(kernel="rbf", nu=.06, gamma=.4).fit(Xn2)
xx, yy = np.meshgrid(np.linspace(-5,5,300), np.linspace(-5,5,300))
Z = oc2.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.figure(figsize=(6,5.6))
plt.contour(xx, yy, Z, levels=[0], colors=[AMBER], linewidths=2.2)
plt.scatter(*Xn2.T, s=16, c=NORMAL, label="Normal (train)")
plt.scatter(*an2.T, s=55, c=ANOM, marker="X", label="Novel")
plt.xticks([]); plt.yticks([]); plt.legend(); plt.title("One-Class SVM boundary"); plt.show()

## 4. Autoencoder: reconstruction error as a score

Train an autoencoder on legitimate transactions only. It learns to rebuild normal points
with tiny error; fraud breaks the learned pattern, so reconstruction MSE spikes. The error is the score.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

Xn = torch.tensor(Xtr[ytr == 0], dtype=torch.float32)      # legitimate only
ae = nn.Sequential(nn.Linear(29,16), nn.ReLU(), nn.Linear(16,8), nn.ReLU(),
                   nn.Linear(8,16), nn.ReLU(), nn.Linear(16,29))
opt, loss_fn = torch.optim.Adam(ae.parameters(), lr=1e-3), nn.MSELoss()

losses = []
for epoch in range(30):
    perm = torch.randperm(len(Xn)); tot = 0.0
    for i in range(0, len(Xn), 512):
        b = Xn[perm[i:i+512]]
        opt.zero_grad(); loss = loss_fn(ae(b), b); loss.backward(); opt.step()
        tot += loss.item() * len(b)
    losses.append(tot / len(Xn))
print(f"train loss {losses[0]:.3f} -> {losses[-1]:.3f}")

with torch.no_grad():
    Xt = torch.tensor(Xte, dtype=torch.float32)
    ae_scores = ((ae(Xt) - Xt) ** 2).mean(1).numpy()
ae_roc, ae_pr, ae_pk = evaluate("Autoencoder", ae_scores)

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(range(1, 31), losses, color=VIOLET, marker="o", ms=3); a1.set_xlabel("Epoch")
a1.set_ylabel("Reconstruction MSE"); a1.set_title("The autoencoder converges")
hi = np.percentile(ae_scores, 99.9)
bins = np.linspace(0, hi, 70)
a2.hist(np.clip(ae_scores[yte==0],0,hi), bins=bins, color=NORMAL, alpha=.85, density=True, label="Legitimate")
a2.hist(np.clip(ae_scores[yte==1],0,hi), bins=bins, color=ANOM, alpha=.7, density=True, label="Fraud")
a2.axvline(np.percentile(ae_scores,95), color="#222", ls="--", label="95th pct")
a2.set_xlabel("Reconstruction error"); a2.set_title("Fraud reconstructs badly"); a2.legend()
plt.tight_layout(); plt.show()

## 5. The honest comparison: ROC vs PR vs precision@k

All three detectors reach ROC-AUC ~0.95, which looks like a solved problem. The
precision-recall panel and precision@k tell the real story under a 0.17% base rate.

In [ ]:
methods = [("Isolation Forest", if_scores, TEAL),
           ("One-Class SVM",   oc_scores, AMBER),
           ("Autoencoder",     ae_scores, VIOLET)]
res = {n: (roc_auc_score(yte,s), average_precision_score(yte,s), prec_at_k(s,yte,k))
       for n,s,_ in methods}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.6))
for n, s, c in methods:
    fpr, tpr, _ = roc_curve(yte, s)
    a1.plot(fpr, tpr, color=c, lw=2, label=f"{n} (AUC {res[n][0]:.3f})")
a1.plot([0,1],[0,1],"--",color="#999"); a1.set_title("ROC: all look excellent")
a1.set_xlabel("FPR"); a1.set_ylabel("TPR"); a1.legend(loc="lower right", fontsize=9)
for n, s, c in methods:
    pr, rc, _ = precision_recall_curve(yte, s)
    a2.plot(rc, pr, color=c, lw=2, label=f"{n} (AP {res[n][1]:.3f})")
a2.axhline(prevalence, ls="--", color="#999", label=f"Random ({prevalence:.4f})")
a2.set_title("PR: the truth"); a2.set_xlabel("Recall"); a2.set_ylabel("Precision")
a2.legend(loc="upper right", fontsize=9)
plt.tight_layout(); plt.show()

print("\nAlways-legitimate baseline accuracy:", f"{(yte==0).mean():.4%}", "(catches 0 fraud)")
for n in res:
    roc, pr, pk = res[n]
    print(f"{n:18s} ROC-AUC {roc:.3f} | PR-AUC {pr:.3f} | P@{k} {pk:.3f}")

## 6. The visually similar mimic (MNIST)

Reconstruction error shines at telling apart lookalikes. Train an autoencoder on
handwritten 3s only, then show it 8s (a 3 with the left side closed). It rebuilds each 8
using the vocabulary of 3s, so the error separates them, without ever training on an 8.
This is the shape of the medical "cancer vs benign mimic" problem.

In [ ]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False)   # downloads ~15 MB once
Xm = mnist.data.astype('float32') / 255.0
ym = mnist.target.astype(int)
NORM_D, MIM_D = 3, 8
g = np.random.default_rng(0)
ni, mi = np.where(ym==NORM_D)[0], np.where(ym==MIM_D)[0]
g.shuffle(ni); g.shuffle(mi)
tr, te_n, te_m = ni[:5000], ni[5000:5800], mi[:800]

torch.manual_seed(0)
img_ae = nn.Sequential(nn.Linear(784,128), nn.ReLU(), nn.Linear(128,32), nn.ReLU(),
                       nn.Linear(32,128), nn.ReLU(), nn.Linear(128,784), nn.Sigmoid())
o2, lf = torch.optim.Adam(img_ae.parameters(), lr=1e-3), nn.MSELoss()
Xtr_i = torch.tensor(Xm[tr])
for _ in range(25):
    p = torch.randperm(len(Xtr_i))
    for i in range(0, len(Xtr_i), 256):
        b = Xtr_i[p[i:i+256]]; o2.zero_grad(); l = lf(img_ae(b), b); l.backward(); o2.step()

with torch.no_grad():
    Xn_i, Xk_i = torch.tensor(Xm[te_n]), torch.tensor(Xm[te_m])
    en = ((img_ae(Xn_i)-Xn_i)**2).mean(1).numpy()
    em = ((img_ae(Xk_i)-Xk_i)**2).mean(1).numpy()
    rec_m = img_ae(Xk_i).numpy()
auc = roc_auc_score(np.r_[np.zeros(len(en)), np.ones(len(em))], np.r_[en, em])
print(f"digit {NORM_D} vs mimic {MIM_D}: reconstruction-error ROC-AUC = {auc:.3f}")

fig, axes = plt.subplots(2, 5, figsize=(10, 4.2))
for j in range(5):
    axes[0,j].imshow(Xm[te_m[j]].reshape(28,28), cmap="gray_r"); axes[0,j].axis("off")
    axes[1,j].imshow(rec_m[j].reshape(28,28), cmap="gray_r"); axes[1,j].axis("off")
axes[0,0].set_ylabel("input 8", rotation=0, labelpad=30); axes[1,0].set_ylabel("rebuilt as 3", rotation=0, labelpad=30)
fig.suptitle("Autoencoder trained on 3s rebuilds 8s as 3-like shapes")
plt.tight_layout(); plt.show()

## Exercises

1. **Contamination.** The Isolation Forest here is an *outlier* detector (it saw fraud in
   training). Refit it on legitimate transactions only and re-evaluate. Does novelty framing help?
2. **Threshold by budget.** Suppose your team can review 100 alerts a day. Threshold each
   detector at its top 100 scores and report the precision. Which detector wins at that budget?
3. **Feature scaling.** Remove the `StandardScaler`. Which of the three detectors degrades most,
   and why? (Hint: axis-aligned splits vs RBF distances vs reconstruction MSE.)
4. **Ensemble.** Average the rank of the three scores. Does the combined ranking beat the best
   single detector on PR-AUC?
5. **Deeper autoencoder.** Add a layer or widen the bottleneck. Does a lower training loss
   translate into a better PR-AUC, or does it start reconstructing fraud too well?